# Programming Assignment 1: Recommender System
### Coure: CS373
#### Developer 1: Jalin A. Brown
#### Developer 2: Sandeep Durga
#### File: 01_data_and_knn.ipynb
#### File: P1_1-2_Setup_CF_KNN.ipynb

### Part One: Data Setup and User-Based Collaborative Filtering

1. Load and preprocess data:
- Create user-item rating matrix.
- Calculate percentage of missing ratings:  Sparsity = (Number of Missing Ratings / Total Possible Ratings) × 100

2. Build K-Nearest Neighbors recommender:
- Find k most similar users for any target user (test different values of k)
- Only consider users with positive similarity scores
- Predict ratings using weighted average
- Handle edge case

In [9]:
import numpy as np                                     # math
import pandas as pd                                    # user-item matrix
from sklearn.metrics.pairwise import cosine_similarity # pairwise similarity

### 1. Load and preprocess data:

In [10]:
# Create user-item rating matrix
user_item_base = pd.read_csv("../data/u1.base", names=["user_id","item_id","rating","timestamp"], sep="\t", )
user_item_test = pd.read_csv("../data/u1.test", names=["user_id","item_id","rating","timestamp"], sep="\t")

# Display info
print(f"\nTraining Data Matrix: {user_item_base.shape}\n")
print(user_item_base.head())
print(f"\nTest Data Matrix: {user_item_test.shape}\n")
print(user_item_test.head())


Training Data Matrix: (80000, 4)

   user_id  item_id  rating  timestamp
0        1        1       5  874965758
1        1        2       3  876893171
2        1        3       4  878542960
3        1        4       3  876893119
4        1        5       3  889751712

Test Data Matrix: (20000, 4)

   user_id  item_id  rating  timestamp
0        1        6       5  887431973
1        1       10       3  875693118
2        1       12       5  878542960
3        1       14       5  874965706
4        1       17       3  875073198


In [11]:
# Cleaning: Exclude possible duplicates && non int vals

# training user-item matrix
train_users = user_item_base["user_id"].astype(int).unique()
train_items = user_item_base["item_id"].astype(int).unique()
user_item_training_matrix = pd.DataFrame(np.nan, index=train_users, columns=train_items)
for row in user_item_base.itertuples(index = False):
    user_item_training_matrix.loc[row.user_id, row.item_id] = row.rating

# test user-item matrix
test_users = user_item_test["user_id"].astype(int).unique()
test_items = user_item_test["item_id"].astype(int).unique()
user_item_test_matrix = pd.DataFrame(np.nan, index=test_users, columns=test_items)
for row in user_item_test.itertuples(index = False):
    user_item_training_matrix.loc[row.user_id, row.item_id] = row.rating

# Display info
print(f"\nTraining User-Item Matrix: {user_item_training_matrix.shape}\n")
print(user_item_training_matrix.head())

print(f"\nTest User-Item Matrix: {user_item_test_matrix.shape}\n")
print(user_item_test_matrix.head())


Training User-Item Matrix: (943, 1682)

   1     2     3     4     5     7     8     9     11    13    ...  1533  \
1   5.0   3.0   4.0   3.0   3.0   4.0   1.0   5.0   2.0   5.0  ...   NaN   
2   4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   4.0  ...   NaN   
3   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   
4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   4.0   NaN  ...   NaN   
5   4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   NaN   

   1536  1543  1557  1561  1562  1563  1565  1582  1586  
1   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
5   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  

[5 rows x 1682 columns]

Test User-Item Matrix: (459, 1410)

   6     10    12    14    17    20    23    24    27    31    ...  1565  \
1   NaN   NaN   

In [12]:
# Analyze sparsity
user_count = user_item_training_matrix.shape[0]
item_count = user_item_training_matrix.shape[1]

mat_size = user_count * item_count
present = user_item_training_matrix.count().sum()
missing = mat_size - present
sparsity = (missing / mat_size) * 100

print(f"sparsity: {sparsity:}\n")

sparsity: 93.69533063577546



### 2. Build KNN Recommender:

#### This next module covers:
- Full similarity space with Nan set to 0
- Only keep users with cosine_similarity > 0 (positive relationship)

Note:
- Positive similarity (0.0, 1.0] indicates similar taste
- Negative similarity [-1, 0.0) indicates opposite taste
- Each index [i,j] is the cosine similarity score between user i and j

In [13]:
# create cosine similarity matrix between all users
users = user_item_training_matrix

# For each user, express ratings as deviation from their personal average.
# If a user didn’t rate an item, treat it as no deviation 0.
user_mean_centered = users.sub(users.mean(axis=1), axis=0).fillna(0.0)
sim = cosine_similarity(user_mean_centered)
user_index = users.index.to_numpy()
cosine_sim_matrix = pd.DataFrame(sim, index= user_index, columns=user_index)

cosine_sim_matrix.head()

,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
1,1.000000,0.043411,0.011051,0.059303,0.134514,0.103373,0.110556,0.180891,0.012253,-0.000621,...,0.025835,-0.047952,0.087224,0.007718,0.074378,0.078714,0.067433,0.028790,-0.031270,0.032123
2,0.043411,1.000000,0.013658,-0.017016,0.035770,0.094503,0.089408,0.055640,0.027294,0.097846,...,0.012853,-0.028798,0.056659,0.197835,0.090009,0.032505,0.015053,-0.017344,0.012068,0.039173
3,0.011051,0.013658,1.000000,-0.059638,0.016037,-0.017158,0.016141,0.041177,-0.010093,0.023856,...,0.001615,0.000658,-0.006888,0.036157,-0.018513,-0.006240,-0.023907,0.034414,-0.009187,0.001489
4,0.059303,-0.017016,-0.059638,1.000000,0.007373,-0.053929,-0.025604,0.136046,0.016082,-0.013588,...,0.011895,0.002174,-0.028000,-0.025021,0.022882,-0.005960,0.279818,0.258594,0.064504,-0.019222
5,0.134514,0.035770,0.016037,0.007373,1.000000,0.038484,0.067874,0.140106,0.010195,0.014335,...,0.070014,-0.070821,0.024278,0.038672,0.093567,0.051782,0.029540,0.036234,0.043318,0.099324


#### Find K most similar users for any target user

(positive similarity only, testing different values of K)

using user 1 as an example

In [17]:
# Select the target user
current_user = 1

# Select a value of K
K_values = [1, 3, 5, 7, 10]

# Extract similarities between this user and all other users
similarities_to_others = cosine_sim_matrix[current_user].copy()
similarities_to_others = similarities_to_others.drop(index=current_user)
pos_sim = similarities_to_others[similarities_to_others > 0.0].sort_values(ascending=False)

print(f"Top similar users for user {current_user} with varying K:\n")
for k in K_values:
    top_k = pos_sim.head(k)
    print(f"K = {k}:")
    print(top_k)
    print("\n")

Top similar users for user 1 with varying K:

K = 1:
773    0.204792
Name: 1, dtype: float64


K = 3:
773    0.204792
868    0.202321
592    0.196592
Name: 1, dtype: float64


K = 5:
773    0.204792
868    0.202321
592    0.196592
880    0.195801
429    0.190661
Name: 1, dtype: float64


K = 7:
773    0.204792
868    0.202321
592    0.196592
880    0.195801
429    0.190661
276    0.187476
916    0.186358
Name: 1, dtype: float64


K = 10:
773    0.204792
868    0.202321
592    0.196592
880    0.195801
429    0.190661
276    0.187476
916    0.186358
222    0.182415
457    0.182253
8      0.180891
Name: 1, dtype: float64




###  Predict ratings using weighted average
Using the weighted similarity formula:
- Select target user (u) and a target item (i) for which the user has not rated yet
- Gather the K most similar neighbor users who have rated the target item
- Compute the weighted average of their ratings using similarity as weight
- If no neighbor has rated that item, handle as an edge case

In [15]:
# Check Target user for an unrated target item
user_item_training_matrix

,1,2,3,4,5,7,8,9,11,13,...,1533,1536,1543,1557,1561,1562,1563,1565,1582,1586
1,5.0,3.0,4.0,3.0,3.0,4.0,1.0,5.0,2.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,2.0,NaN,4.0,5.0,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,5.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
942,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# Choose target user and unrated target item
target_user = 2
target_item = 2
k_value = 5

# Gather k most similar neighbors who have rated the target item
pos_sim = cosine_sim_matrix[current_user].drop(index=current_user)
pos_sim = pos_sim[pos_sim > 0.0].sort_values(ascending=False)
top_k = pos_sim.head(k_value)

numerator = 0.0
denominator = 0.0

for neighbor, sim in top_k.items():
    neighbor_rating = user_item_training_matrix.loc[neighbor, target_item]
    if not np.isnan(neighbor_rating):
        numerator += neighbor_rating * sim
        denominator += abs(sim)
if denominator == 0:
    print(f"No similar users for user {target_user} rated {target_item}.")
    predicted_rating = np.nan
else:
    predicted_rating = numerator / denominator
    print(f"Using K value {k_value}, the predicted rating for user {target_user} on item {target_item} is {predicted_rating:.2f}.")

Using K value 5, the predicted rating for user 2 on item 2 is 2.75.
